### Master Seminar Physical Geography 2026
## Elevational Temperature Gradient and seasonal variability in Greenland 2010-2026
#### Lea Pietrek & Miriam Erdler
---
#### 💡 Aim of the Project: characterize how elevational temperature gradients across Greenland have evolved using PROMICE Station Pairs


---
---

## Settings

In [2]:
# install necessary libraries
!pip install pandas numpy matplotlib seaborn scipy pymannkendall jupyter


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# import necessary libraries
import pandas as pd #reading and manipulating data
import numpy as np #numerical operations
from pathlib import Path #handling file paths

## Data Cleaning

Paths: This section defines all folder paths used in the workflow. Using relative paths instead of local machine paths ensures that the notebook can be executed on different computers without modifying the code.

In [25]:
# Define paths and constants

# project root = current folder
PROJECT_ROOT = Path.cwd()

# paths
RAW_DATA = PROJECT_ROOT / "Data" / "Data_RAW"

PROCESSED_DATA = PROJECT_ROOT / "Data" / "Data_Processed"

CLEAN_DATA = PROCESSED_DATA / "Cleaned_Stations"

# create output folder
CLEAN_DATA.mkdir(parents=True, exist_ok=True)

Let's insepct the data! This code will check for the existing CSV Files.

In [5]:
# List all CSV files in the raw data directory
# The workflow is designed to automatically detect all PROMICE station files, 
# making it scalable and reducing the need for manual file selection.

csv_files = list(RAW_DATA.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files")

for file in csv_files:
    print(file.name)

Found 18 CSV files
KAN_L_month.csv
KAN_U_month.csv
KPC_L_month.csv
KPC_U_month.csv
NUK_L_month.csv
NUK_U_month.csv
QAS_L_month.csv
QAS_U_month.csv
SCO_L_month.csv
SCO_U_month.csv
TAS_L_month.csv
TAS_U_month.csv
THU_L_month.csv
THU_U_month.csv
UPE_L_month.csv
UPE_U_month.csv
ZAC_L_month.csv
ZAC_U_month.csv


Let's continue inspecting the data and have a closer look at the dataframe

In [19]:
# Read a sample file to check the structure

sample_file = csv_files[0]

df = pd.read_csv(sample_file)

df.head()

,time,p_u,t_u,rh_u,rh_u_wrt_ice_or_water,qh_u,wspd_u,wdir_u,wspd_x_u,wspd_y_u,...,gps_alt,lat,lon,alt,batt_v,t_rad,t_i,rh_i_wrt_ice_or_water,wdir_i,wspd_y_i
0,2008-09-01,923.9882,-2.5934,79.9399,81.6629,2.8476,3.3134,133.9854,1.8320,-1.7682,...,676.17,67.097312,-49.931913,678.73,13.45,-2.0476,-2.5962,81.6650,134.9669,-1.7649
1,2008-10-01,924.2335,-8.8390,76.7142,81.4251,1.8167,4.5009,136.0388,2.4764,-2.5678,...,675.85,67.097292,-49.932110,678.70,13.25,-8.9245,-8.8353,82.3774,134.6686,-2.6032
2,2008-11-01,924.7323,-10.8112,81.3601,88.1435,1.6097,4.3467,133.9104,2.3021,-2.2162,...,674.28,67.097272,-49.932309,678.66,12.82,-11.2204,-10.8295,88.6081,131.1321,-2.2228
3,2008-12-01,916.8134,-15.4373,71.2094,78.6264,1.0833,5.7925,140.0709,3.3923,-4.0529,...,670.38,67.097251,-49.932509,678.63,12.55,-15.9078,-15.4285,83.5962,140.3030,-3.9943
4,2009-01-01,912.9546,-13.5028,62.6707,68.1504,1.1081,5.3631,129.9273,3.7522,-3.1404,...,678.00,67.097231,-49.932711,678.60,12.55,-14.0442,-13.4724,73.9714,124.2769,-3.1638


In [7]:
# Check column names
df.columns

Index(['time', 'p_u', 't_u', 'rh_u', 'rh_u_wrt_ice_or_water', 'qh_u', 'wspd_u',
       'wdir_u', 'wspd_x_u', 'wspd_y_u', 'dsr', 'dsr_cor', 'usr', 'usr_cor',
       'albedo', 'dlr', 'ulr', 'cc', 't_surf', 'dlhf_u', 'dshf_u', 'z_boom_u',
       'z_boom_cor_u', 'z_stake', 'z_stake_cor', 'z_pt', 'z_pt_cor',
       'z_surf_combined', 'z_ice_surf', 'snow_height', 'rainfall_u',
       'rainfall_cor_u', 't_i_1', 't_i_2', 't_i_3', 't_i_4', 't_i_5', 't_i_6',
       't_i_7', 't_i_8', 'd_t_i_1', 'd_t_i_2', 'd_t_i_3', 'd_t_i_4', 'd_t_i_5',
       'd_t_i_6', 'd_t_i_7', 'd_t_i_8', 't_i_10m', 'tilt_x', 'tilt_y', 'rot',
       'gps_lat', 'gps_lon', 'gps_alt', 'lat', 'lon', 'alt', 'batt_v', 't_rad',
       't_i', 'rh_i_wrt_ice_or_water', 'wdir_i', 'wspd_y_i'],
      dtype='str')

##### Extract Metadata from File Names:
PROMICE file names contain important metadata information.
Example:

`KAN_L_month.csv`

From this naming structure we extract:
- station ID (`KAN`)
- station level (`L` = lower station, `U` = upper station)
- temporal resolution (`month`)
  
This metadata will later help identify station pairs and distinguish between elevation levels.

This function extracts these three pieces of information automatically from any filename,
so the code works for all 18 files without manual input.

In [32]:
# Function to extract station information from file name
def extract_station_info(file_path):
    """
    KAN_L_month.csv → ("KAN", "L", "month")
    """
    parts = file_path.stem.split("_")
    return parts[0], parts[1], parts[2]

### Define Cleaning Function
This function applies all cleaning steps to a single PROMICE station file.

**Steps performed:**
1. Extract station metadata from the filename
2. Load the raw CSV file
3. Replace PROMICE no-data codes (`-999`, `-9999`) with `NaN`
4. Keep only the columns relevant for this analysis
5. Parse the `time` column as a proper datetime format
6. Filter rows to the study period 2010–2026
7. Replace physically impossible temperatures with `NaN`
   (values outside –70°C to +30°C cannot occur in Greenland)
8. Remove duplicate rows and sort chronologically
9. Add metadata columns (`station_id`, `site`, `position`, `resolution`)
10. Add a season label to each row (DJF / MAM / JJA / SON)

In [ ]:
# Season lookup: month → season label. Each month belongs to exactly one season, so we can use a simple dictionary.
SEASON_MAP = {
    1:"DJF", 2:"DJF", 3:"MAM", 4:"MAM",  5:"MAM",
    6:"JJA", 7:"JJA", 8:"JJA", 9:"SON", 10:"SON",
    11:"SON", 12:"DJF"
}

# Function to clean one PROMICE file
def clean_promice_file(file_path):
    """
    Load and clean one raw PROMICE CSV.
    Always reads from RAW_DATA — never modifies the original file.
    """
    station_id, station_level, temporal_resolution = extract_station_info(file_path)

    # Load raw file
    df = pd.read_csv(file_path)

    # Replace PROMICE no-data codes with NaN
    df = df.replace([-999, -9999], np.nan)

    # Keep only columns needed for analysis
    columns_to_keep = ["time", "gps_lat", "gps_lon", "gps_alt", "t_u", "rh_u"]
    df = df[[c for c in columns_to_keep if c in df.columns]]

    # Parse time as datetime
    df["time"] = pd.to_datetime(df["time"])

    # Filter to study period 2010–2026
    df = df[df["time"].dt.year.between(2010, 2026)]

    # Remove physically impossible temperatures
    if "t_u" in df.columns:
        df.loc[~df["t_u"].between(-70, 30), "t_u"] = np.nan

    # Remove duplicates, sort by time
    df = df.drop_duplicates().sort_values("time").reset_index(drop=True)

    # Add metadata from filename
    df["station_id"] = f"{station_id}_{station_level}"
    df["site"]       = station_id
    df["position"]   = station_level
    df["resolution"] = temporal_resolution

    # Add season label
    df["season"] = df["time"].dt.month.map(SEASON_MAP)

    return df

### Clean and Save All 18 Station Files
The cleaning function is applied to every raw file automatically.

Each cleaned file is saved individually to `Data_Processed/Cleaned_Stations/`
using the same naming structure as the raw file, with `_clean` added.

Example: `KAN_L_month.csv` → `KAN_L_month_clean.csv`

In [ ]:
# Apply cleaning to every raw file and save to Cleaned_Stations 
# The output file name is derived from the input file name, with a "_clean" suffix for clarity.
for file in csv_files:

    df = clean_promice_file(file)

    station_id, station_level, temporal_resolution = extract_station_info(file)

    out_name = f"{station_id}_{station_level}_{temporal_resolution}_clean.csv"
    df.to_csv(CLEAN_DATA / out_name, index=False)

    print(f"  {file.name:25s} → {len(df):3d} rows | t_u NaN: {df['t_u'].isna().sum()}")

print(f"\n✅ All files cleaned and saved to Cleaned_Stations/")

  KAN_L_month.csv           → 196 rows | t_u NaN: 1
  KAN_U_month.csv           → 196 rows | t_u NaN: 12
  KPC_L_month.csv           → 195 rows | t_u NaN: 54
  KPC_U_month.csv           → 195 rows | t_u NaN: 30
  NUK_L_month.csv           → 190 rows | t_u NaN: 32
  NUK_U_month.csv           → 196 rows | t_u NaN: 21
  QAS_L_month.csv           → 196 rows | t_u NaN: 7
  QAS_U_month.csv           → 196 rows | t_u NaN: 33
  SCO_L_month.csv           → 195 rows | t_u NaN: 13
  SCO_U_month.csv           → 196 rows | t_u NaN: 29
  TAS_L_month.csv           → 195 rows | t_u NaN: 59
  TAS_U_month.csv           →  68 rows | t_u NaN: 14
  THU_L_month.csv           → 181 rows | t_u NaN: 40
  THU_U_month.csv           → 181 rows | t_u NaN: 14
  UPE_L_month.csv           → 195 rows | t_u NaN: 18
  UPE_U_month.csv           → 195 rows | t_u NaN: 27
  ZAC_L_month.csv           → 196 rows | t_u NaN: 23
  ZAC_U_month.csv           → 192 rows | t_u NaN: 53

✅ All files cleaned and saved to Cleaned_Statio

##### Data Quality Check

In [29]:
# Show % missing per variable per station
missing = {}

for file in sorted(CLEAN_DATA.glob("*.csv")):
    df = pd.read_csv(file)
    missing[file.stem] = df.isna().mean() * 100

pd.DataFrame(missing).round(1)

,KAN_L_month_clean,KAN_U_month_clean,KPC_L_month_clean,KPC_U_month_clean,NUK_L_month_clean,NUK_U_month_clean,QAS_L_month_clean,QAS_U_month_clean,SCO_L_month_clean,SCO_U_month_clean,TAS_L_month_clean,TAS_U_month_clean,THU_L_month_clean,THU_U_month_clean,UPE_L_month_clean,UPE_U_month_clean,ZAC_L_month_clean,ZAC_U_month_clean
time,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
gps_lat,18.9,35.2,25.6,17.9,15.3,11.2,13.8,25.5,45.1,80.1,52.8,64.7,49.2,54.1,11.3,20.5,8.2,66.7
gps_lon,18.9,36.7,25.6,17.9,16.3,11.2,13.8,25.5,45.1,80.1,52.8,64.7,49.2,54.1,11.3,20.5,8.2,66.7
gps_alt,18.9,35.2,25.6,17.9,15.3,11.2,12.8,25.0,45.1,80.1,52.8,64.7,49.2,54.1,11.3,21.0,8.2,66.7
t_u,0.5,6.1,27.7,15.4,16.8,10.7,3.6,16.8,6.7,14.8,30.3,20.6,22.1,7.7,9.2,13.8,11.7,27.6
rh_u,5.6,3.6,25.1,12.8,23.2,6.6,2.0,18.4,1.5,10.2,24.1,14.7,20.4,3.9,7.7,10.3,6.1,25.0
station_id,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
site,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
position,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
resolution,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Build Stations Pairs + compute Temperature Gradient

In [30]:
# Load all 18 cleaned files into one DataFrame
all_stations = pd.concat(
    [pd.read_csv(f, parse_dates=["time"]) for f in sorted(CLEAN_DATA.glob("*.csv"))],
    ignore_index=True
)

# Split into upper and lower stations
upper = all_stations[all_stations["position"] == "U"]
lower = all_stations[all_stations["position"] == "L"]

# Merge upper + lower on site and time → one row per site per month
pairs = pd.merge(
    upper, lower,
    on=["site", "time", "season", "resolution"],
    suffixes=("_upper", "_lower")
)

# Elevation difference between stations
pairs["elev_diff_m"] = pairs["gps_alt_upper"] - pairs["gps_alt_lower"]

# Temperature gradient in °C per 100m — main analysis variable
pairs["gradient_per100m"] = (
    (pairs["t_u_upper"] - pairs["t_u_lower"]) / pairs["elev_diff_m"]
) * 100

# Drop rows where gradient could not be computed
pairs = pairs.dropna(subset=["gradient_per100m"])

print(f"✅ {len(pairs)} paired records | {pairs['site'].nunique()} station pairs")
print(f"\nRecords and mean gradient per site:")
print(pairs.groupby("site")["gradient_per100m"].agg(["count", "mean"]).round(3))

✅ 776 paired records | 9 station pairs

Records and mean gradient per site:
      count   mean
site              
KAN     123 -0.709
KPC     124 -0.737
NUK     134 -0.676
QAS     120 -0.713
SCO      24 -0.450
TAS      11 -0.668
THU      56 -1.032
UPE     142 -0.504
ZAC      42 -0.316


Save the final station pairs

In [31]:
# Save — this is the only file the analysis section needs
out_path = PROCESSED_DATA / "station_pairs_monthly.csv"
pairs.to_csv(out_path, index=False)

print(f"✅ Saved: {out_path.name}")
print(f"   {len(pairs)} rows | {pairs['site'].nunique()} station pairs")
print(f"   Columns: {pairs.columns.tolist()}")

✅ Saved: station_pairs_monthly.csv
   776 rows | 9 station pairs
   Columns: ['time', 'gps_lat_upper', 'gps_lon_upper', 'gps_alt_upper', 't_u_upper', 'rh_u_upper', 'station_id_upper', 'site', 'position_upper', 'resolution', 'season', 'gps_lat_lower', 'gps_lon_lower', 'gps_alt_lower', 't_u_lower', 'rh_u_lower', 'station_id_lower', 'position_lower', 'elev_diff_m', 'gradient_per100m']
